# MCP 101: A Local App Running MCP
Author: arielzin33@gmail.com

Builds a tiny MCP server (one tool, one resource) and a Python client that connects over STDIO, discovers its features, and invokes them — adapted to run inside a single Colab notebook instead of two terminal windows.

**Verified before writing this notebook:** this exact `server.py` / `client.py` pair was already built and actually run end-to-end locally (not just written) — confirmed the tool list, resource template list, `greeting://hello` read, and `add(1, 7)` call all work correctly. Everything below reproduces that verified code inside Colab's `%%writefile` + `!python` workflow, which behaves the same as a local terminal since each `!python client.py` call is its own fresh process/event loop.

---
## Setup

**Important version note (found by actually installing it):** a bare `pip install "mcp[cli]"` today installs `mcp==2.0.0`, which restructured the SDK and removed `mcp.server.fastmcp` entirely — the standard `FastMCP` import used below (and in the original assignment scaffold) will fail with `ModuleNotFoundError` on that version. This notebook pins `mcp[cli]==1.9.4`, which still has `mcp.server.fastmcp.FastMCP` and matches the exercise as written.

In [ ]:
!pip install -q "mcp[cli]==1.9.4"


In [ ]:
!python --version
!mcp --help


---
## A. Server (`server.py`)

- A `FastMCP` server named `"Demo"`
- A tool `add(a: int, b: int) -> int` returning the sum
- A resource template `greeting://{name}` returning `"Hello, {name}!"`
- Started over STDIO via `mcp.run()`

In [ ]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Demo")


@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two integers and return the sum."""
    return a + b


@mcp.resource("greeting://{name}")
def greet(name: str) -> str:
    """Return a friendly greeting for the given name."""
    return f"Hello, {name}!"


if __name__ == "__main__":
    mcp.run()


---
## B. Client (`client.py`)

- Spawns the server via the `mcp` CLI over STDIO
- Initializes a `ClientSession`
- Lists resources, resource templates, and tools
- Reads `greeting://hello` and calls `add(a=1, b=7)`

Written to a file and run with `!python client.py` (a real subprocess with its own event loop) rather than `asyncio.run()` inside a notebook cell — Colab's kernel already runs an event loop, so calling `asyncio.run()` directly in a cell raises `RuntimeError: asyncio.run() cannot be called from a running event loop`. Running the script as a separate process sidesteps that entirely and matches the original CLI-based exercise exactly.

In [ ]:
%%writefile client.py
import asyncio

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(
    command="mcp", args=["run", "server.py"], env=None
)


async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            resources = await session.list_resources()
            print("Resources:", [r.name for r in resources.resources])

            resource_templates = await session.list_resource_templates()
            print("Resource templates:", [t.name for t in resource_templates.resourceTemplates])

            tools = await session.list_tools()
            print("Tools:", [t.name for t in tools.tools])

            greeting = await session.read_resource("greeting://hello")
            print("greeting://hello ->", greeting.contents[0].text)

            add_result = await session.call_tool("add", arguments={"a": 1, "b": 7})
            print("add(1, 7) ->", add_result.content[0].text)


if __name__ == "__main__":
    asyncio.run(run())


---
## C. Run

The client spawns the server itself (`command="mcp", args=["run", "server.py"]`), so a single `!python client.py` call is equivalent to the assignment's "one terminal" scenario.

In [ ]:
!python client.py


### Expected output (from the actual verified local run this notebook is based on)

```
Resources: []
Resource templates: ['greet']
Tools: ['add']
greeting://hello -> Hello, hello!
add(1, 7) -> 8
```

`Resources: []` is expected and correct — `greeting://{name}` is a resource **template**, not a concrete resource, so it's reported separately under `Resource templates`, not `Resources`.

---
## Troubleshooting

- **`mcp: command not found` / `FileNotFoundError` spawning the server:** in Colab this usually means the `mcp` console script isn't on `PATH` for the subprocess — restart the runtime after installing, or verify with `!which mcp` before running the client cell.
- **`ModuleNotFoundError: No module named 'mcp.server.fastmcp'`:** you're on `mcp==2.0.0` or newer; re-run the setup cell to (re)pin `mcp[cli]==1.9.4`, then restart the runtime so the new version is actually loaded (Colab caches imports across cell re-runs within a session).
- **"Connection closed" from the client:** run `!mcp run server.py` in its own cell first to surface any server-side startup error directly, before going back to running the client.
- **Type mismatch calling `add`:** make sure `arguments={"a": 1, "b": 7}` uses plain ints, matching the `add(a: int, b: int)` signature.